# Introducción a NLP: preprocesamiento de texto y clasificación con RNN y LSTM

En la clase 06 las secuencias eran eventos de un log, con un vocabulario de 29 símbolos. En esta clase trabajamos con texto en lenguaje natural. Antes de entrenar un modelo hay que convertir cada reseña en una secuencia de enteros: limpieza, tokenización, vocabulario y padding. Luego comparamos tres clasificadores de sentimiento: un modelo que promedia los embeddings de las palabras (ignora el orden), una RNN y una LSTM.

## Introducción

### Objetivos

1. **Implementar un pipeline de preprocesamiento de texto**: limpieza, tokenización, normalización (stemming y lematización) y construcción del vocabulario.
2. **Convertir texto en secuencias de índices de largo fijo**, con padding y truncado.
3. **Entrenar un modelo base que promedia embeddings** y entender por qué ignora el orden de las palabras.
4. **Comparar una RNN y una LSTM sobre secuencias largas** y relacionar el resultado con el problema del gradiente visto en la clase 06.

### Contenido

1. Configuración y carga del dataset IMDB.
2. Limpieza de bajo y alto nivel: tokenización, stop words, stemming y lematización.
3. Vocabulario, conversión a índices, padding y truncado.
4. `Dataset` y `DataLoader`.
5. Modelo base: promedio de embeddings.
6. RNN y LSTM.
7. Comparación en validación y evaluación final en test.

### Sobre el conjunto de datos

Usamos el dataset de reseñas de películas de IMDB, publicado por [Maas et al. (2011)](https://aclanthology.org/P11-1015/) en la Universidad de Stanford y disponible [aquí](https://ai.stanford.edu/~amaas/data/sentiment/). Contiene 50,000 reseñas en inglés etiquetadas como positivas o negativas, divididas en 25,000 para entrenamiento y 25,000 para test. Las dos clases están balanceadas. Es un conjunto de referencia para **análisis de sentimiento**: predecir si una reseña es positiva o negativa a partir de su texto.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchinfo import summary  # resumen de capas, formas y parámetros

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split  # división estratificada train/val

import os
import tarfile  # descomprimir el dataset (.tar.gz)
import urllib.request  # descargar el dataset
import re  # expresiones regulares para la limpieza de texto
from collections import Counter  # contar la frecuencia de cada palabra (vocabulario)
from itertools import chain  # unir las listas de tokens de todas las reseñas en una sola

# NLTK (Natural Language Toolkit): biblioteca clásica de procesamiento de lenguaje natural
import nltk
from nltk.corpus import stopwords  # listas de stop words por idioma
from nltk.stem import PorterStemmer  # stemming: recorta sufijos con reglas fijas
from nltk.stem import WordNetLemmatizer  # lematización: forma base según el diccionario WordNet
from nltk.corpus import wordnet  # constantes de categoría gramatical (ADJ, VERB, NOUN, ADV)

from utils import (
    train,
    accuracy,
    plot_training,
    model_classification_report,
)

In [ ]:
# Fijamos la semilla para que los resultados sean reproducibles
SEED = 23

torch.manual_seed(SEED)
# cuDNN elige algoritmos deterministas (mismo resultado en cada corrida, algo más lento)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
from utils import get_device, get_num_workers

DEVICE = get_device()  # cuda > mps > xpu > cpu
# Linux: mitad de los núcleos disponibles, entre 1 y 8; Windows/macOS: 0 (ver docstring)
NUM_WORKERS = get_num_workers()

print(f"Usando {DEVICE}")
print(f"Usando {NUM_WORKERS} workers")

In [ ]:
BATCH_SIZE = 512  # tamaño del batch

## Carga de datos

Descargamos el [dataset IMDB](https://ai.stanford.edu/~amaas/data/sentiment/) y lo cargamos en un DataFrame de Pandas.

El archivo contiene dos carpetas, `train` y `test`, cada una con las subcarpetas `pos` y `neg`. Cada reseña está en un archivo de texto separado.

In [ ]:
DATA_PATH = "data"
URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
tar_path = os.path.join(DATA_PATH, "aclImdb_v1.tar.gz")
extracted_path = os.path.join(DATA_PATH, "aclImdb")

os.makedirs(DATA_PATH, exist_ok=True)

# Descargamos el archivo comprimido (unos 80 MB) solo si no está en disco
if not os.path.exists(tar_path):
    print("Descargando el dataset...")
    urllib.request.urlretrieve(URL, tar_path)

# Extraemos solo la primera vez: son unos 100,000 archivos de texto
if not os.path.exists(extracted_path):
    print("Extrayendo archivos...")
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=DATA_PATH, filter="data")

print("Contenido:", os.listdir(extracted_path))

In [ ]:
TRAIN_PATH = os.path.join(extracted_path, "train")
TEST_PATH = os.path.join(extracted_path, "test")

In [ ]:
# Cargar el dataset de IMDB desde archivos locales
def load_imdb_data(base_directory):
    data = []
    for label in ["pos", "neg"]:
        folder = os.path.join(base_directory, label)
        # sorted: el orden de os.listdir depende del sistema de archivos
        for file in sorted(os.listdir(folder)):
            with open(os.path.join(folder, file), "r", encoding="utf-8") as f:
                data.append((f.read(), 1 if label == "pos" else 0))  # 1 para positivo, 0 para negativo
    return pd.DataFrame(data, columns=["review", "sentiment"])

train_df = load_imdb_data(TRAIN_PATH)
test_df = load_imdb_data(TEST_PATH)

print(f"Entrenamiento: {len(train_df)} reseñas | Test: {len(test_df)} reseñas")
print(train_df["sentiment"].value_counts())

In [ ]:
# Mostramos una reseña tal como viene en el dataset. Elegimos una corta (menos de 600 caracteres)
# para que se lea completa, y que tenga etiquetas HTML (<br />), que luego hay que limpiar
is_short_with_html = train_df["review"].str.contains("<br />") & (train_df["review"].str.len() < 600)
EXAMPLE_IDX = train_df[is_short_with_html].index[0]

print(f"Sentimiento: {train_df.loc[EXAMPLE_IDX, 'sentiment']}\n")
print(train_df.loc[EXAMPLE_IDX, "review"])

## Preprocesamiento de texto

Una red trabaja con tensores numéricos, así que el preprocesamiento convierte cada reseña en una secuencia de índices de largo fijo:

<img src="https://raw.githubusercontent.com/Catedra-IA/taller-deep-learning-letra/main/assets/nlp_pipeline.png" width="900" style="background:white; display: block; margin-left: auto; margin-right: auto;"/>

1. Limpieza de bajo nivel: quitar las etiquetas HTML, el texto entre corchetes, los caracteres especiales (conservamos la puntuación básica) y los espacios repetidos.
2. Limpieza de alto nivel: pasar a minúsculas, separar en tokens y reducir cada palabra a su raíz (stemming) o a su forma base (lematización). También analizamos la eliminación de stop words.
3. Vocabulario: asignar un índice a cada palabra frecuente. Las palabras fuera del vocabulario se mapean a un token especial.
4. Padding y truncado: llevar todas las secuencias al mismo largo.

### Limpieza del texto de bajo nivel

Eliminamos las etiquetas HTML, el texto entre corchetes, los caracteres especiales y los espacios en blanco innecesarios. Para eso usamos expresiones regulares de la [biblioteca `re`](https://docs.python.org/3/library/re.html).

In [ ]:
def strip_html_tags(text):
    """Elimina etiquetas HTML"""
    pattern = r"<.*?>"
    return re.sub(pattern, "", text)

def remove_between_square_brackets(text):
    """Elimina texto entre corchetes cuadrados (ej: [Spoiler], [Citation needed])"""
    pattern = r"\[[^\]]*\]"
    return re.sub(pattern, "", text)

def remove_special_characters(text, keep_punctuation=True):
    """
    Elimina caracteres especiales

    Args:
        text: texto a limpiar
        keep_punctuation: si True, conserva puntuación básica (.,!?;:)
    """
    if keep_punctuation:
        # Conservamos letras, números, espacios y puntuación común
        pattern = r"[^a-zA-Z0-9\s.,!?;:\-']"
    else:
        # Solo letras, números y espacios
        pattern = r"[^a-zA-Z0-9\s]"

    return re.sub(pattern, "", text)

def remove_additional_whitespace(text):
    """Elimina espacios en blanco múltiples y espacios alrededor de puntuación"""
    # Primero eliminamos espacios múltiples
    text = re.sub(r"\s{2,}", " ", text)
    # Eliminamos espacios antes de puntuación
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    # Aseguramos un espacio después de puntuación (si no hay ya)
    text = re.sub(r"([.,!?;:])([^\s])", r"\1 \2", text)
    # Eliminamos espacios al inicio y final
    return text.strip()

def low_level_text_cleaning(text, keep_punctuation=True):
    """
    Limpieza completa de texto

    Args:
        text: texto a limpiar
        keep_punctuation: si True, conserva puntuación (recomendado para NLP)
    """
    text = strip_html_tags(text)
    text = remove_between_square_brackets(text)
    text = remove_special_characters(text, keep_punctuation)
    text = remove_additional_whitespace(text)
    return text


# Texto de prueba
testing_text = "<p>Hello    World! [Spoiler] The villain is the butler! [End of spoiler] <br> See you.</p>"
print(f"Original: {testing_text}")
print(f"Limpio:   {low_level_text_cleaning(testing_text)}")

In [ ]:
# Aplicamos la limpieza a los conjuntos de entrenamiento y test
train_df["review_low_level_cleaned"] = train_df["review"].apply(low_level_text_cleaning)
test_df["review_low_level_cleaned"] = test_df["review"].apply(low_level_text_cleaning)

### Limpieza del texto de alto nivel

Pasamos el texto a minúsculas, para que "Movie" y "movie" sean el mismo token, y analizamos las **stop words**: palabras muy frecuentes ("a", "the", "is") que en muchas tareas aportan poca información. La biblioteca `nltk` provee una lista para inglés.

> **Nota**: si la descarga falla con `429: Too Many Requests`, descargar el paquete manualmente con `python -m nltk.downloader stopwords`.

In [ ]:
# Descargamos stopwords de nltk
nltk.download("stopwords")
all_stopwords = set(stopwords.words("english"))

print(f"{len(all_stopwords)} stop words, por ejemplo: {sorted(all_stopwords)[:12]}")
print(f"¿'not' es una stop word? {'not' in all_stopwords}")

In [ ]:
def remove_stop_words(full_text_line):
    # Eliminamos stopwords
    tokens = full_text_line.split()
    tokens = [token for token in tokens if token not in all_stopwords]
    return " ".join(tokens)

def to_lower_case(full_text_line):
    # Convertimos a minúsculas
    return full_text_line.lower()

# Texto de prueba
testing_text = "This movie is not good and I did not like it"
print(f"Original:                     {testing_text}")
print(f"Minúsculas y sin stop words:  {remove_stop_words(to_lower_case(testing_text))}")

Para análisis de sentimiento, quitar las stop words de NLTK es contraproducente: la lista incluye negaciones como *not*, *no* y *don't*, y una reseña negativa pasa a parecer positiva. Por eso en esta clase no quitamos las stop words. La función queda disponible para los ejercicios.

#### Lematización y stemming

- La **lematización** convierte cada palabra a su forma base o lema. Por ejemplo, "corriendo", "corre" y "corrió" se convierten en "correr". Para elegir el lema correcto necesita conocer la categoría gramatical de la palabra (*part of speech*, POS): "meeting" es "meet" si es verbo y "meeting" si es sustantivo.
- El **stemming** elimina sufijos con reglas fijas para obtener una raíz, que no siempre es una palabra válida. Por ejemplo, "corriendo", "corre" y "corrió" se convierten en "corr". Es mucho más rápido que la lematización, pero menos preciso.

Las dos técnicas reducen el vocabulario: variantes de una misma palabra comparten un índice y su embedding se entrena con más ejemplos.

In [ ]:
# Tokenizador (nltk.word_tokenize): separa la puntuación y las contracciones
nltk.download("punkt")
nltk.download("punkt_tab")

# Lematizador (WordNetLemmatizer)
nltk.download("wordnet")

# Etiquetador POS (nltk.pos_tag) y su conjunto de etiquetas simplificado ("universal")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("universal_tagset")

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

Separar el texto en tokens no es tan sencillo como hacer un `split(" ")`: hay que separar la puntuación y las contracciones. El tokenizador de NLTK `nltk.word_tokenize()` resuelve estos casos.

In [ ]:
test_texts = [
    "Hello, world! This is a test.",
    "It's a beautiful day, isn't it?",
    "Dr. Smith went to Washington D.C. on Jan. 5th.",
    "The quick brown fox jumps over the lazy dog."
]

for testing_text in test_texts:
    tokens = nltk.word_tokenize(testing_text)  # separamos en tokens
    print(f"Original: {testing_text}")
    print(f"Tokens:   {tokens}\n")

In [ ]:
# pos_tag con tagset="universal" devuelve etiquetas como NOUN, VERB, ADJ o ADV;
# WordNetLemmatizer usa sus propias constantes, así que las traducimos
UNIVERSAL_TO_WORDNET = {
    "ADJ": wordnet.ADJ,  # adjetivo
    "VERB": wordnet.VERB,  # verbo
    "NOUN": wordnet.NOUN,  # sustantivo
    "ADV": wordnet.ADV,  # adverbio
}

def get_wordnet_pos(universal_tag):
    return UNIVERSAL_TO_WORDNET.get(universal_tag, wordnet.NOUN)  # por defecto, sustantivo

text = "The children were running faster than their friends and watched better movies."

# Tokenizamos y etiquetamos cada token con su categoría gramatical
tokens = nltk.word_tokenize(text.lower())
pos_tags = nltk.pos_tag(tokens, tagset="universal")

print(f"{'Token':<12} {'POS':<6} {'Lema':<12} {'Stem':<12}")
print("-" * 44)
for token, pos in pos_tags:
    if token.isalpha():  # solo palabras, ignoramos la puntuación
        lemma = lemmatizer.lemmatize(token, get_wordnet_pos(pos))
        print(f"{token:<12} {pos:<6} {lemma:<12} {stemmer.stem(token):<12}")

In [ ]:
def lemmatize(text):
    tokens = nltk.word_tokenize(text)
    pos_tags = nltk.pos_tag(tokens, tagset="universal")
    # Lematizamos cada token con su categoría gramatical
    return " ".join(lemmatizer.lemmatize(token, get_wordnet_pos(pos)) for token, pos in pos_tags)

def stem(text):
    tokens = nltk.word_tokenize(text)
    return " ".join(stemmer.stem(token) for token in tokens)

def high_level_text_cleaning(text, normalize=stem, drop_stop_words=False):
    text = to_lower_case(text)
    if drop_stop_words:
        text = remove_stop_words(text)
    return normalize(text)  # stem o lemmatize; ambas tokenizan y unen los tokens con espacios

print(f"Original: {text}")
print(f"Lema:     {high_level_text_cleaning(text, normalize=lemmatize)}")
print(f"Stem:     {high_level_text_cleaning(text, normalize=stem)}")

In [ ]:
testing_text = "Why waste time saying a lot of words when a few words do the trick?"
print(f"Lema: {high_level_text_cleaning(testing_text, normalize=lemmatize)}")
print(f"Stem: {high_level_text_cleaning(testing_text, normalize=stem)}")

<img src="https://media1.tenor.com/m/IsYdPRq7bjcAAAAC/why-waste-time-when-few-word-do-trick.gif"/>

[Kevin's Small Talk - The Office US](https://www.youtube.com/watch?v=_K-L9uhsBLM)

In [ ]:
# Usamos stemming: es mucho más rápido que la lematización, que etiqueta cada palabra
# con su categoría gramatical. Probar lemmatize queda como ejercicio.
train_df["review_clean"] = train_df["review_low_level_cleaned"].apply(high_level_text_cleaning)
test_df["review_clean"] = test_df["review_low_level_cleaned"].apply(high_level_text_cleaning)

print(train_df.loc[EXAMPLE_IDX, "review_clean"])

### División en entrenamiento y validación

El vocabulario se construye solo con el conjunto de entrenamiento. Si incluyera las reseñas de validación, las palabras que solo aparecen ahí tendrían un índice propio en lugar del token de fuera de vocabulario, y la validación dejaría de simular datos nunca vistos. Por eso separamos antes de construir el vocabulario, con `train_test_split` estratificado como en la clase 06. El test se usa **una sola vez**, al final.

In [ ]:
train_df, val_df = train_test_split(
    train_df, test_size=0.2, random_state=SEED, stratify=train_df["sentiment"]
)
print(f"Entrenamiento: {len(train_df)} | Validación: {len(val_df)} | Test: {len(test_df)}")

### Construcción del vocabulario

El vocabulario asigna un índice único a cada palabra. Conservamos las `MAX_VOCAB_SIZE` palabras más frecuentes que aparecen al menos `min_freq` veces. El resto se mapea al token `<OOV>` (*out of vocabulary*). El índice 0 se reserva para el token de relleno `<PAD>`.

In [ ]:
OOV_TOKEN = "<OOV>"  # palabra fuera del vocabulario
PAD_TOKEN = "<PAD>"  # relleno para llevar las secuencias al mismo largo
MAX_VOCAB_SIZE = 20_000  # cantidad de palabras más frecuentes que conservamos
SEQUENCE_LENGTH = 200  # mediana del largo de las reseñas (en tokens): se trunca cerca de la mitad
EMBEDDING_DIM = 100  # tamaño del vector de cada palabra

In [ ]:
def make_vocab(all_texts, max_vocab_size, min_freq=5):
    # Contamos la frecuencia de cada palabra
    counts = Counter(chain(*(all_texts.str.split())))
    counts = {word: freq for word, freq in counts.items() if freq >= min_freq}

    # Ordenamos las palabras por frecuencia y nos quedamos con las max_vocab_size palabras más frecuentes
    vocab = sorted(counts, key=counts.get, reverse=True)[:max_vocab_size]
    vocab.append(OOV_TOKEN)  # Añadimos el token OOV al final

    # Mapa de palabras a índices
    vocab_to_int = {word: ii for ii, word in enumerate(vocab, 1)}
    vocab_to_int[PAD_TOKEN] = 0  # Añadimos el token PAD al principio

    return vocab_to_int

# Texto de prueba: sin pasar a minúsculas, "The" y "the" son entradas distintas
testing_text1 = "The quick brown fox jumps over the lazy dog"
testing_text2 = "Then the quick brown fox jumps over the lazy dog"
print(f"Vocab: {make_vocab(pd.Series([testing_text1, testing_text2]), 100, 1)}")

In [ ]:
word_to_index = make_vocab(train_df["review_clean"], MAX_VOCAB_SIZE)
VOCAB_SIZE = len(word_to_index)  # hasta MAX_VOCAB_SIZE palabras (con min_freq=5 quedan algunas menos) + <OOV> + <PAD>
print(f"Tamaño del vocabulario: {VOCAB_SIZE}")

### De texto a secuencia de índices

Convertimos cada reseña en una lista de índices y la llevamos a `SEQUENCE_LENGTH` tokens:

- **Truncado**: si la reseña es más larga, conservamos los últimos tokens (`keep='last'`). El final de una reseña suele contener la conclusión del autor.
- **Left padding**: si es más corta, completamos con `<PAD>` a la izquierda. Como en la clase 06, así las palabras reales quedan en los últimos pasos, los más cercanos al estado oculto final que usa el clasificador.

In [ ]:
def get_review_features(review_text, word_to_idx):
    """
    Convierte un texto en una lista de índices basados en el vocabulario.
    """
    # El texto ya está tokenizado: los tokens están separados por espacios
    return [
        word_to_idx.get(word, word_to_idx[OOV_TOKEN]) for word in review_text.split()
    ]

def truncate_sequence(sequence, max_length, keep="last"):
    """
    Trunca la secuencia manteniendo las primeras o últimas palabras

    Args:
        sequence: lista de tokens/índices
        max_length: longitud máxima
        keep: 'first' o 'last'
    """
    if len(sequence) <= max_length:
        return sequence

    if keep == "last":
        return sequence[-max_length:]  # Últimas palabras
    else:
        return sequence[:max_length]  # Primeras palabras

def left_pad_features(review_ints, seq_length, pad_value=0, truncate_keep="last"):
    """
    Aplica padding a la izquierda a una secuencia de índices para que todas las secuencias tengan la misma longitud.
    """
    # Truncar si es más largo que seq_length
    review_ints = truncate_sequence(review_ints, seq_length, keep=truncate_keep)

    # Padding a la izquierda
    padding = [pad_value] * (seq_length - len(review_ints))
    return padding + review_ints

def get_review_representation(review_text, word_to_idx, max_sequence_length):
    """
    Convierte el texto de entrada en una representación de secuencia con padding a la izquierda.
    """
    review_ints = get_review_features(review_text, word_to_idx)
    return left_pad_features(review_ints, max_sequence_length)

# Texto de prueba
testing_text = "The quick brown fox jumps <br> over the lazy dog ! ThisWordIsOOV."
print(f"Original:            {testing_text}")
testing_text = low_level_text_cleaning(testing_text)
print(f"Limpieza bajo nivel: {testing_text}")
testing_text = high_level_text_cleaning(testing_text)
print(f"Limpieza alto nivel: {testing_text}")
testing_text = get_review_representation(testing_text, word_to_index, 15)
print(f"Índices:             {testing_text}")

## Dataset y DataLoader

`IMDBDataset` recibe las **reseñas preprocesadas**, las etiquetas y el vocabulario, y devuelve para cada reseña su secuencia de índices y su etiqueta.

In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, reviews, labels, vocab, max_sequence_length):
        self.reviews = reviews.tolist()  # listas: acceso por posición (0, 1, 2, ...)
        self.labels = labels.tolist()
        self.vocab = vocab
        self.max_sequence_length = max_sequence_length

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, idx):
        # Convertir texto a representación con padding
        indices = get_review_representation(self.reviews[idx], self.vocab, self.max_sequence_length)
        return torch.tensor(indices, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

In [ ]:
train_dataset = IMDBDataset(train_df["review_clean"], train_df["sentiment"], word_to_index, SEQUENCE_LENGTH)
val_dataset = IMDBDataset(val_df["review_clean"], val_df["sentiment"], word_to_index, SEQUENCE_LENGTH)
test_dataset = IMDBDataset(test_df["review_clean"], test_df["sentiment"], word_to_index, SEQUENCE_LENGTH)

In [ ]:
x, y = train_dataset[0]
print(f"Índices: {x.shape} | Etiqueta: {y}")
print(x)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# Parte 1: modelo base sin recurrencia

## Modelo base

### `nn.Embedding`

En la clase 06 usamos [`nn.Embedding`](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html) como tabla de búsqueda para 29 eventos. Con texto la idea es la misma, pero la tabla tiene una fila por palabra del vocabulario: con unas 20,000 palabras y `EMBEDDING_DIM = 100`, la matriz es de `VOCAB_SIZE` × 100. La capa no tiene bias, así que

$$
\text{Parámetros} = \text{tamaño del vocabulario} \times \text{tamaño del embedding}
$$

Es la capa con más parámetros del modelo. Con `padding_idx`, la fila de `<PAD>` vale cero y no se actualiza durante el entrenamiento.

In [ ]:
embedding_layer = nn.Embedding(VOCAB_SIZE, EMBEDDING_DIM, padding_idx=word_to_index[PAD_TOKEN])
word_indices = torch.tensor([0, 1, 2, 3, 4, 5])
embeddings = embedding_layer(word_indices)

print(f"Forma: {embeddings.shape}")  # (6, EMBEDDING_DIM): un vector por índice
print(embeddings[:, :5])  # primeras 5 componentes; la fila de <PAD> (índice 0) es cero

### Arquitectura

1. `nn.Embedding`: cada índice se convierte en un vector, `(batch, 200)` → `(batch, 200, 100)`.
2. Promedio sobre la dimensión de la secuencia: un único vector por reseña, `(batch, 100)`.
3. `nn.Linear` con ReLU (capa oculta).
4. `nn.Linear` de salida con dos logits (negativa, positiva), que van a `nn.CrossEntropyLoss` como en la clase 06.

El promedio no depende del orden: "not bad, good" y "not good, bad" producen el mismo vector. Este modelo es una **bolsa de embeddings** (*bag of embeddings*): ve qué palabras aparecen, pero no en qué orden. Es la línea base que un modelo secuencial tiene que superar.

<small>Nota: el promedio incluye las posiciones de `<PAD>`, cuyo embedding es cero. En una reseña corta el vector promedio queda escalado hacia cero. Promediar solo las posiciones reales es una mejora sencilla (ver Ejercicios).</small>

In [ ]:
# Salida esperada de summary (usarla para verificar la implementación):
#
# SentimentModel                           [512, 2]                  --
# ├─Embedding: 1-1                         [512, 200, 100]           1,955,300
# ├─Linear: 1-2                            [512, 512]                51,712
# ├─Linear: 1-3                            [512, 2]                  1,026
# Total params: 2,008,038

class SentimentModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes=2):
        super(SentimentModel, self).__init__()
        # TODO: definir las capas como atributos:
        #   1. self.embedding: nn.Embedding de vocab_size a embedding_dim, con padding_idx=word_to_index[PAD_TOKEN]
        #   2. self.fc1: nn.Linear de embedding_dim a hidden_dim
        #   3. self.out: nn.Linear de hidden_dim a num_classes
        pass

    def forward(self, x):
        # x: (batch, SEQUENCE_LENGTH) con índices enteros
        # TODO:
        #   1. embedding -> (batch, SEQUENCE_LENGTH, embedding_dim)
        #   2. promedio sobre la secuencia (dim=1) -> (batch, embedding_dim)
        #   3. fc1 + ReLU -> (batch, hidden_dim)
        #   4. out -> logits (batch, num_classes); sin activación (CrossEntropyLoss incluye la softmax)
        pass

summary(
    SentimentModel(VOCAB_SIZE, EMBEDDING_DIM, hidden_dim=512),
    input_size=(BATCH_SIZE, SEQUENCE_LENGTH),
    dtypes=[torch.long],
)

### Entrenamiento

In [ ]:
CRITERION = nn.CrossEntropyLoss()
TARGET_NAMES = ["negativa", "positiva"]

LR = 1e-3
EPOCHS = 20
PATIENCE = 5

In [ ]:
torch.manual_seed(SEED)
base_model = SentimentModel(VOCAB_SIZE, EMBEDDING_DIM, hidden_dim=512).to(DEVICE)
base_optimizer = optim.Adam(base_model.parameters(), lr=LR)

base_train_errors, base_val_errors = train(
    base_model,
    optimizer=base_optimizer,
    criterion=CRITERION,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    do_early_stopping=True,
    patience=PATIENCE,
    epochs=EPOCHS,
)

In [ ]:
plot_training(base_train_errors, base_val_errors, mark_min=True)

In [ ]:
model_classification_report(base_model, val_loader, DEVICE, 2, digits=4, target_names=TARGET_NAMES)

# Parte 2: modelos recurrentes

En la clase 06 vimos que una RNN recorre la secuencia paso a paso y resume lo visto en el estado oculto, y que la influencia de una entrada sobre el último estado decae con la distancia. Allí las secuencias tenían 50 eventos; en esta clase tienen 200 palabras. Usamos la misma arquitectura **many-to-one**: la entrada es la secuencia de palabras y la salida, una etiqueta para toda la reseña.

## RNN

### RNN apilada

Usamos dos capas recurrentes apiladas (`num_layers=2`): la segunda capa recibe como entrada la secuencia de estados ocultos de la primera.

<img src="https://d2l.ai/_images/deep-rnn.svg" width="400" style="background:white; display: block; margin-left: auto; margin-right: auto;"/>

[`nn.RNN`](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html) devuelve dos tensores:

- `output`: los estados ocultos de la **última capa** en todos los pasos, `(batch, secuencia, hidden_size)`.
- `hidden`: el último estado de **cada capa**, `(num_layers, batch, hidden_size)`. `hidden[-1]` es el último estado de la última capa, el que usa el clasificador.

In [ ]:
rnn = nn.RNN(input_size=EMBEDDING_DIM, hidden_size=32, num_layers=2, batch_first=True)
tensor = torch.randn(BATCH_SIZE, SEQUENCE_LENGTH, EMBEDDING_DIM)

output, hidden = rnn(tensor)
print(f"Output shape: {output.shape} (batch_size, seq_length, hidden_size)")
print(f"Hidden shape: {hidden.shape} (num_layers, batch_size, hidden_size)")
# El último paso de output es el último estado de la última capa
print(f"output[:, -1] == hidden[-1]: {torch.allclose(output[:, -1], hidden[-1])}")

### Arquitectura

Respecto del modelo base, el promedio se reemplaza por la RNN y la clasificación se hace con `hidden[-1]`. Agregamos dropout sobre los embeddings y antes de la capa de salida como regularización (clase 03).

In [ ]:
# Salida esperada de summary (usarla para verificar la implementación):
#
# SentimentRNN                             [512, 2]                  --
# ├─Embedding: 1-1                         [512, 200, 100]           1,955,300
# ├─Dropout: 1-2                           [512, 200, 100]           --
# ├─RNN: 1-3                               [512, 200, 128]           62,464
# ├─Linear: 1-4                            [512, 256]                33,024
# ├─Dropout: 1-5                           [512, 256]                --
# ├─Linear: 1-6                            [512, 2]                  514
# Total params: 2,051,302

class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, n_layers=2, dropout=0.5, num_classes=2):
        super(SentimentRNN, self).__init__()
        # TODO: definir las capas como atributos:
        #   1. self.embedding: nn.Embedding de vocab_size a embedding_dim, con padding_idx=word_to_index[PAD_TOKEN]
        #   2. self.rnn: nn.RNN de embedding_dim a hidden_dim, con num_layers=n_layers y batch_first=True
        #   3. self.dropout: nn.Dropout(dropout)
        #   4. self.fc1: nn.Linear de hidden_dim a hidden_dim * 2
        #   5. self.fc2: nn.Linear de hidden_dim * 2 a num_classes
        pass

    def forward(self, x):
        # x: (batch, SEQUENCE_LENGTH) con índices enteros
        # TODO:
        #   1. embedding + dropout -> (batch, SEQUENCE_LENGTH, embedding_dim)
        #   2. rnn -> output, hidden; hidden tiene forma (n_layers, batch, hidden_dim)
        #   3. último estado de la última capa: hidden[-1] -> (batch, hidden_dim)
        #   4. fc1 + ReLU + dropout -> (batch, 2 * hidden_dim)
        #   5. fc2 -> logits (batch, num_classes); sin activación
        pass

HIDDEN_DIM = 128

summary(
    SentimentRNN(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM),
    input_size=(BATCH_SIZE, SEQUENCE_LENGTH),
    dtypes=[torch.long],
)

### Entrenamiento

In [ ]:
torch.manual_seed(SEED)
rnn_model = SentimentRNN(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, n_layers=2, dropout=0.5).to(DEVICE)
rnn_optimizer = optim.Adam(rnn_model.parameters(), lr=LR)

rnn_train_errors, rnn_val_errors = train(
    rnn_model,
    optimizer=rnn_optimizer,
    criterion=CRITERION,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    do_early_stopping=True,
    patience=PATIENCE,
    epochs=EPOCHS,
)

In [ ]:
plot_training(rnn_train_errors, rnn_val_errors, mark_min=True)

In [ ]:
model_classification_report(rnn_model, val_loader, DEVICE, 2, digits=4, target_names=TARGET_NAMES)

La RNN queda claramente por debajo del modelo base (en nuestra ejecución, 74% contra 87% de accuracy en validación), aunque el modelo base ni siquiera usa el orden de las palabras. La pérdida de entrenamiento también baja muy poco, así que la red tampoco logra ajustar bien los datos de entrenamiento.

La causa es la que vimos en la clase 06: la influencia de una entrada sobre el último estado decae de forma exponencial con la distancia. Con 200 pasos, el gradiente que llega a las primeras palabras es prácticamente nulo, y la red aprende casi solo de las últimas. La LSTM, que vemos a continuación, se diseñó para este problema.

## LSTM

La [LSTM](https://d2l.ai/chapter_recurrent-modern/lstm.html) (*Long Short-Term Memory*, Hochreiter y Schmidhuber, 1997) agrega al estado oculto $h_t$ un **estado de celda** $c_t$ que se actualiza de forma aditiva:

$$
c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \qquad\qquad h_t = o_t \odot \tanh(c_t)
$$

- $f_t$ (**compuerta de olvido**) decide qué parte de la memoria anterior se conserva.
- $i_t$ (**compuerta de entrada**) decide cuánto de la información nueva $\tilde{c}_t$ se escribe.
- $o_t$ (**compuerta de salida**) decide qué parte de la memoria se expone en $h_t$.

Las compuertas son vectores con valores entre 0 y 1 (sigmoides) que se calculan a partir de $x_t$ y $h_{t-1}$. En la RNN el estado se recalcula por completo en cada paso: el gradiente atraviesa una multiplicación por $W_{hh}$ y una $\tanh$ por paso, y decae de forma exponencial. En la LSTM, cuando $f_t \approx 1$, el gradiente pasa por $c_t$ casi sin atenuarse: la red puede aprender a conservar información durante muchos pasos.

<img src="https://d2l.ai/_images/lstm-3.svg" width="500" style="background:white; display: block; margin-left: auto; margin-right: auto;"/>

El artículo de Christopher Olah [*Understanding LSTM Networks*](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) y el video de StatQuest [*Long Short-Term Memory (LSTM), Clearly Explained*](https://www.youtube.com/watch?v=YCzL96nL7j0) recorren las compuertas paso a paso.

<small>La LSTM mitiga el desvanecimiento del gradiente. Para gradientes que explotan se usa *gradient clipping* (`torch.nn.utils.clip_grad_norm_`) dentro del bucle de entrenamiento.</small>

In [ ]:
lstm = nn.LSTM(input_size=EMBEDDING_DIM, hidden_size=32, num_layers=2, batch_first=True)
tensor = torch.randn(BATCH_SIZE, SEQUENCE_LENGTH, EMBEDDING_DIM)

# Además de output y hidden, la LSTM devuelve el último estado de celda de cada capa
output, (hidden, cell) = lstm(tensor)
print(f"Output shape: {output.shape} (batch_size, seq_length, hidden_size)")
print(f"Hidden shape: {hidden.shape} (num_layers, batch_size, hidden_size)")
print(f"Cell shape:   {cell.shape} (num_layers, batch_size, hidden_size)")

In [ ]:
# Salida esperada de summary (usarla para verificar la implementación):
#
# SentimentLSTM                            [512, 2]                  --
# ├─Embedding: 1-1                         [512, 200, 100]           1,955,300
# ├─Dropout: 1-2                           [512, 200, 100]           --
# ├─LSTM: 1-3                              [512, 200, 128]           249,856
# ├─Linear: 1-4                            [512, 256]                33,024
# ├─Dropout: 1-5                           [512, 256]                --
# ├─Linear: 1-6                            [512, 2]                  514
# Total params: 2,238,694

class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, n_layers=2, dropout=0.5, num_classes=2):
        super(SentimentLSTM, self).__init__()
        # TODO: igual que SentimentRNN, pero con self.lstm = nn.LSTM(...) en lugar de nn.RNN
        pass

    def forward(self, x):
        # x: (batch, SEQUENCE_LENGTH) con índices enteros
        # TODO: igual que SentimentRNN, pero la LSTM devuelve output, (hidden, cell);
        #   para clasificar usamos hidden[-1]
        pass

summary(
    SentimentLSTM(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM),
    input_size=(BATCH_SIZE, SEQUENCE_LENGTH),
    dtypes=[torch.long],
)

In [ ]:
torch.manual_seed(SEED)
lstm_model = SentimentLSTM(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, n_layers=2, dropout=0.5).to(DEVICE)
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=LR)

lstm_train_errors, lstm_val_errors = train(
    lstm_model,
    optimizer=lstm_optimizer,
    criterion=CRITERION,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    do_early_stopping=True,
    patience=PATIENCE,
    epochs=EPOCHS,
)

In [ ]:
plot_training(lstm_train_errors, lstm_val_errors, mark_min=True)

In [ ]:
model_classification_report(lstm_model, val_loader, DEVICE, 2, digits=4, target_names=TARGET_NAMES)

## Comparación

Comparamos los tres modelos en validación. El test queda reservado para el modelo elegido.

In [ ]:
models = {"Promedio de embeddings": base_model, "RNN": rnn_model, "LSTM": lstm_model}
val_errors = {"Promedio de embeddings": base_val_errors, "RNN": rnn_val_errors, "LSTM": lstm_val_errors}

for name, model in models.items():
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{name:<24} parámetros: {n_params:>9,} | accuracy en validación: {accuracy(model, val_loader, DEVICE):.4f}")

plt.figure(figsize=(8, 4))
for name, errors in val_errors.items():
    plt.plot(range(1, len(errors) + 1), errors, marker="o", label=name)
plt.xlabel("Época")
plt.ylabel("Pérdida de validación")
plt.title("Pérdida de validación por modelo")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

En nuestra ejecución, la RNN queda unos 13 puntos por debajo de los otros dos modelos porque no logra propagar el gradiente a lo largo de 200 pasos. La LSTM alcanza la menor pérdida de validación, aunque su curva es mucho más irregular, y en accuracy iguala al modelo base.

El promedio de embeddings resulta una línea base muy competitiva. En una reseña, palabras como *great* o *awful* ya determinan el sentimiento en la mayoría de los casos. El orden importa en negaciones y contrastes (*"not bad"*, *"good actors, terrible script"*), que son una fracción menor de los ejemplos. Un modelo más complejo se justifica solo si supera a esta línea base.

## Evaluación final en test

Elegimos la LSTM, el modelo con menor pérdida de validación (en accuracy iguala al modelo base), y la evaluamos una sola vez sobre el conjunto de test.

In [ ]:
model_classification_report(
    lstm_model, test_loader, DEVICE, 2, digits=4, target_names=TARGET_NAMES, do_confusion_matrix=True
)

## Conclusiones

- El preprocesamiento convierte texto en tensores: limpieza, tokenización, normalización (stemming o lematización), un vocabulario construido solo con el conjunto de entrenamiento, y padding y truncado a un largo fijo.
- Las decisiones de preprocesamiento dependen de la tarea: la lista de stop words de NLTK incluye negaciones, que en análisis de sentimiento cambian el significado.
- Un modelo que promedia embeddings ignora el orden de las palabras y, aun así, es una línea base difícil de superar en IMDB.
- Una RNN simple no logra aprender con secuencias de 200 pasos, porque el gradiente se desvanece.
- La LSTM agrega un estado de celda que se actualiza de forma aditiva y compuertas que controlan qué se conserva. Así el gradiente llega a pasos lejanos y la red alcanza el rendimiento del modelo base.

## Ejercicios

1. **Stop words sin negaciones**: sacar de `all_stopwords` las negaciones (*not*, *no*, *nor* y las contracciones con *n't*) y preprocesar con `drop_stop_words=True`. Comparar en validación con el modelo base y con la LSTM.
2. **Lematización**: preprocesar con `normalize=lemmatize`. ¿Cuánto tarda respecto del stemming? ¿Cambia el tamaño del vocabulario o el resultado?
3. **Promedio sin padding**: en el modelo base, promediar solo las posiciones reales. Pista: la máscara `x != 0` indica qué posiciones no son `<PAD>`.
4. **GRU**: reemplazar `nn.LSTM` por [`nn.GRU`](https://d2l.ai/chapter_recurrent-modern/gru.html), que tiene la misma interfaz que `nn.RNN`. ¿Qué cambia en la cantidad de parámetros y en el resultado?
5. **LSTM bidireccional**: usar `bidirectional=True`. ¿Qué forma tiene `hidden`? Concatenar los últimos estados de las dos direcciones (`hidden[-2]` y `hidden[-1]`) antes de la capa lineal. ¿Tiene sentido el left padding en este caso?
6. **Largo de secuencia**: probar `SEQUENCE_LENGTH = 100` y `400`, y `truncate_keep="first"`. ¿Qué parte de la reseña aporta más información?

#### Lectura adicional

- Maas et al. (2011), [*Learning Word Vectors for Sentiment Analysis*](https://aclanthology.org/P11-1015/): el trabajo que publicó el dataset de IMDB.
- Hochreiter y Schmidhuber (1997), [*Long Short-Term Memory*](https://doi.org/10.1162/neco.1997.9.8.1735): el artículo original de la LSTM.
- Olah (2015), [*Understanding LSTM Networks*](https://colah.github.io/posts/2015-08-Understanding-LSTMs/): explicación visual de las compuertas.
- Bird, Klein y Loper, [*Natural Language Processing with Python*, cap. 3](https://www.nltk.org/book/ch03.html): tokenización, stemming y lematización con NLTK.
- Zhang et al., [*Dive into Deep Learning*, cap. 10](https://d2l.ai/chapter_recurrent-modern/index.html): LSTM, GRU, RNN apiladas y bidireccionales.